# LastFM — audit ketat dan eksperimen utama

> Salinan publik ini mempertahankan kode dan output eksperimen yang tersimpan. Preview baris mentah disembunyikan dari output cell agar data tidak ikut didistribusikan. Dataset mentah, file sequence/timestamp, dan checkpoint tidak disertakan. Sel training akan menjalankan training penuh jika dijalankan ulang; jangan memilih “Run all” hanya untuk membaca hasil.

> Sel audit tambahan setelah bagian hasil utama belum dieksekusi pada sumber notebook; gunakan hanya output yang memang tersimpan sebagai bukti.


# Reproduksi BSARec LastFM — versi audit ketat

**Penting untuk melanjutkan run yang sudah ada:** jangan gunakan “Run All” dan jangan jalankan sel training pada bagian E/F. Jalankan hanya sel tambahan setelah G.1, mulai dari arsip Drive, agar enam hasil training yang sudah tersimpan tidak terlatih ulang.


Notebook ini mempertahankan enam run yang sudah selesai dan menambahkan audit preprocessing independen: urutan sequence dan mapping ID dibandingkan langsung dengan prosedur LastFM S3Rec yang dirujuk, invariant 5-core diperiksa, serta hash data dan mapping disimpan. Audit baru tidak melatih ulang model.

Konfigurasi BSARec mengikuti Table 6 paper LastFM: `alpha=0.9`, `c=3`, satu attention head, dan `lr=0.001`. SASRec dijalankan sebagai baseline dengan data, split, dan evaluasi yang sama. Selisih paper di bawah adalah deskriptif: paper melaporkan satu nilai, sedangkan notebook ini merata-ratakan tiga seed.

**Catatan waktu:** `user_taggedartists-timestamps.dat` mencatat waktu tag artis, bukan waktu scrobble/mendengarkan lagu. BSARec/SASRec standar tidak memakai timestamp. Berkas waktu dan gap disimpan sebagai metadata analisis.

**Batas reproduksibilitas:** run training yang sudah terekam tidak mencatat commit Git dan hash sumber secara eksplisit. Audit manifest merekam commit yang tersedia saat sel audit dijalankan; itu membuktikan run lama hanya jika dijalankan pada runtime/repo yang sama.


## A. Setup Colab dan kode resmi

In [1]:
from pathlib import Path
import os, sys, subprocess, zipfile, urllib.request
import torch

ROOT = Path('/content')
REPO = ROOT / 'BSARec'
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/yehjin-shin/BSARec.git', str(REPO)], check=True)
SRC = REPO / 'src'
assert (SRC / 'main.py').exists(), f'main.py tidak ditemukan di {SRC}'
os.chdir(SRC)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scipy', 'tqdm', 'pandas'], check=True)
GPU_AVAILABLE = torch.cuda.is_available()
print('Repo:', REPO)
print('GPU tersedia:', GPU_AVAILABLE)
if GPU_AVAILABLE:
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Runtime tanpa GPU; training tetap berjalan, tetapi bisa lebih lambat.')

Repo: /content/BSARec
GPU tersedia: True
GPU: Tesla T4


## B. Unduh dan periksa data HetRec Last.fm 2K

In [ ]:
DATA_DIR = ROOT / 'lastfm_raw'
DATA_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = DATA_DIR / 'hetrec2011-lastfm-2k.zip'
DATA_URL = 'https://files.grouplens.org/datasets/hetrec2011/hetrec2011-lastfm-2k.zip'
if not ZIP_PATH.exists():
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(DATA_DIR)
raw_matches = list(DATA_DIR.rglob('user_taggedartists-timestamps.dat'))
assert raw_matches, 'user_taggedartists-timestamps.dat tidak ditemukan dalam arsip.'
RAW_PATH = raw_matches[0]
print('Raw file:', RAW_PATH)
print('Contoh baris:', RAW_PATH.read_text(encoding='utf-8', errors='replace').splitlines()[:3])

## C. Preprocessing berpasangan item dan timestamp

Mengikuti prosedur LastFM yang dirujuk BSARec/S3Rec: simpan kemunculan pertama setiap pasangan user-artis dalam urutan file, urutkan sequence per user berdasarkan timestamp (sort stabil untuk tie), terapkan iterative 5-core, lalu petakan ID berdasarkan urutan kemunculan. Ini menjaga item dan timestamp tetap berpasangan saat deduplikasi dan filtering.

`LastFM.txt` adalah file input model. `LastFM_time.txt` berisi timestamp sumber dalam ms pada posisi item yang sama. `LastFM_gap.txt` berisi gap antar-event dalam ms; elemen pertama tiap user bernilai 0 karena tidak ada event sebelumnya.

In [3]:
import pandas as pd
from collections import Counter, OrderedDict

raw = pd.read_csv(RAW_PATH, sep='\t')
raw.columns = [str(c).strip() for c in raw.columns]
assert {'userID', 'artistID', 'timestamp'}.issubset(raw.columns), raw.columns.tolist()
raw = raw[['userID', 'artistID', 'timestamp']].dropna().copy()
raw['timestamp'] = pd.to_numeric(raw['timestamp'], errors='coerce')
raw = raw.dropna(subset=['timestamp']).copy()
raw['timestamp'] = raw['timestamp'].astype('int64')

# Deduplikasi pasangan user-artis sesuai kemunculan pertama pada file sumber.
user_events = OrderedDict()
seen_artists = {}
for user, artist, timestamp in raw.itertuples(index=False, name=None):
    if user not in user_events:
        user_events[user] = []
        seen_artists[user] = set()
    if artist not in seen_artists[user]:
        user_events[user].append((artist, int(timestamp)))
        seen_artists[user].add(artist)

# Sort stabil per user: timestamp tie mempertahankan urutan file sumber.
for user in user_events:
    user_events[user].sort(key=lambda pair: pair[1])

# Iterative user-5-core dan artist-5-core, sambil mempertahankan timestamp.
while True:
    interactions_before = sum(len(seq) for seq in user_events.values())
    user_events = OrderedDict(
        (u, seq) for u, seq in user_events.items() if len(seq) >= 5
    )
    artist_counts = Counter(
        artist for seq in user_events.values() for artist, _ in seq
    )
    user_events = OrderedDict(
        (u, [(artist, ts) for artist, ts in seq if artist_counts[artist] >= 5])
        for u, seq in user_events.items()
    )
    user_events = OrderedDict(
        (u, seq) for u, seq in user_events.items() if len(seq) >= 5
    )
    interactions_after = sum(len(seq) for seq in user_events.values())
    if interactions_after == interactions_before:
        break

# Pemetaan 1-based berdasarkan urutan kemunculan, bukan urutan alfabetis.
user2id = {user: i + 1 for i, user in enumerate(user_events)}
artist2id = {}
item_sequences = OrderedDict()
time_sequences = OrderedDict()
gap_sequences = OrderedDict()

for user, pairs in user_events.items():
    uid = user2id[user]
    item_ids, timestamps = [], []
    for artist, timestamp in pairs:
        if artist not in artist2id:
            artist2id[artist] = len(artist2id) + 1
        item_ids.append(artist2id[artist])
        timestamps.append(timestamp)
    gaps = [0] + [timestamps[i] - timestamps[i - 1] for i in range(1, len(timestamps))]
    item_sequences[uid] = item_ids
    time_sequences[uid] = timestamps
    gap_sequences[uid] = gaps

DATA_OUT = SRC / 'data'
DATA_OUT.mkdir(parents=True, exist_ok=True)
ITEM_FILE = DATA_OUT / 'LastFM.txt'
TIME_FILE = DATA_OUT / 'LastFM_time.txt'
GAP_FILE = DATA_OUT / 'LastFM_gap.txt'

def write_parallel_files(item_path, time_path, gap_path):
    with item_path.open('w', encoding='utf-8') as f_item, \
         time_path.open('w', encoding='utf-8') as f_time, \
         gap_path.open('w', encoding='utf-8') as f_gap:
        for uid, items in item_sequences.items():
            times = time_sequences[uid]
            gaps = gap_sequences[uid]
            assert len(items) == len(times) == len(gaps)
            f_item.write(str(uid) + ' ' + ' '.join(map(str, items)) + '\n')
            f_time.write(str(uid) + ' ' + ' '.join(map(str, times)) + '\n')
            f_gap.write(str(uid) + ' ' + ' '.join(map(str, gaps)) + '\n')

write_parallel_files(ITEM_FILE, TIME_FILE, GAP_FILE)

n_users = len(user2id)
n_artists = len(artist2id)
n_interactions = sum(map(len, item_sequences.values()))
seq_lengths = [len(seq) for seq in item_sequences.values()]
print(f'Users: {n_users:,} | Artists: {n_artists:,} | Interactions: {n_interactions:,}')
print(f'Average sequence: {sum(seq_lengths)/len(seq_lengths):.2f} | min: {min(seq_lengths)} | max: {max(seq_lengths)}')
print('Model input:', ITEM_FILE)
print('Aligned timestamps:', TIME_FILE)
print('Aligned gaps:', GAP_FILE)
assert (n_users, n_artists, n_interactions) == (1090, 3646, 52551), 'Stats differ from BSARec Table 5; stop and inspect preprocessing.'

Users: 1,090 | Artists: 3,646 | Interactions: 52,551
Average sequence: 48.21 | min: 5 | max: 899
Model input: /content/BSARec/src/data/LastFM.txt
Aligned timestamps: /content/BSARec/src/data/LastFM_time.txt
Aligned gaps: /content/BSARec/src/data/LastFM_gap.txt


## D. Validasi file, alignment, dan timestamp

In [4]:
item_lines = ITEM_FILE.read_text(encoding='utf-8').splitlines()
time_lines = TIME_FILE.read_text(encoding='utf-8').splitlines()
gap_lines = GAP_FILE.read_text(encoding='utf-8').splitlines()
assert len(item_lines) == len(time_lines) == len(gap_lines) == n_users

for item_line, time_line, gap_line in zip(item_lines, time_lines, gap_lines):
    item_parts = item_line.split()
    time_parts = time_line.split()
    gap_parts = gap_line.split()
    assert item_parts[0] == time_parts[0] == gap_parts[0]
    assert len(item_parts[1:]) == len(time_parts[1:]) == len(gap_parts[1:])

# Timestamp sumber adalah ms. Konversi ms -> hari dengan 86,400,000.
import numpy as np
all_gaps_ms = np.concatenate([
    np.diff(np.asarray(times, dtype=np.int64))
    for times in time_sequences.values()
    if len(times) > 1
])
print('Item/time/gap row alignment: PASSED')
print('Gap <= 0 (ms):', int((all_gaps_ms <= 0).sum()))
print('Median positive gap (days):', float(np.median(all_gaps_ms[all_gaps_ms > 0]) / 86_400_000))
print('P90 positive gap (days):', float(np.percentile(all_gaps_ms[all_gaps_ms > 0], 90) / 86_400_000))
print('Interpretation: tag-event gaps, not listening/scrobble gaps.')

Item/time/gap row alignment: PASSED
Gap <= 0 (ms): 44273
Median positive gap (days): 31.0
P90 positive gap (days): 153.04166666666666
Interpretation: tag-event gaps, not listening/scrobble gaps.


## D.1 Audit ketat terhadap preprocessing rujukan

Sel ini merekonstruksi langkah LastFM dari `data_process.py` S3Rec (deduplikasi artis per user pada kemunculan pertama, stable sort timestamp, iterative 5-core, lalu ID 1-based menurut urutan traversal). Hasilnya dibandingkan **per user, per item, dan per ID**, bukan hanya statistik agregat. Sel ini juga mengecek monotonic timestamp, duplikasi, 5-core, alignment item/time/gap, dan menyimpan mapping serta SHA-256.

Jika assertion gagal, jangan pakai metrik untuk klaim reproduksi sebelum penyebabnya ditelusuri. Sel ini hanya memproses data dan menulis manifest; tidak menjalankan training.


In [5]:
from collections import Counter, OrderedDict
from copy import deepcopy
from importlib.metadata import version, PackageNotFoundError
import hashlib, json, platform, subprocess
import numpy as np

# Bangun ulang dari baris sumber sebagai string, seperti parser referensi.
ref_raw = pd.read_csv(
    RAW_PATH, sep='\t', dtype={'userID': str, 'artistID': str, 'tagID': str, 'timestamp': str}
)
assert {'userID', 'artistID', 'timestamp'}.issubset(ref_raw.columns)
ref_pairs = OrderedDict()
ref_seen = {}
for user, artist, timestamp in ref_raw[['userID', 'artistID', 'timestamp']].itertuples(index=False, name=None):
    if user not in ref_pairs:
        ref_pairs[user] = []
        ref_seen[user] = set()
    if artist not in ref_seen[user]:
        ref_pairs[user].append((artist, int(timestamp)))
        ref_seen[user].add(artist)
for user in ref_pairs:
    ref_pairs[user].sort(key=lambda pair: pair[1])  # Python sort stabil; tie menjaga urutan file

ref_user_items = OrderedDict((u, [artist for artist, _ in pairs]) for u, pairs in ref_pairs.items())

def _reference_core_counts(user_items):
    user_counts = Counter()
    item_counts = Counter()
    for user, items in user_items.items():
        for item in items:
            user_counts[user] += 1
            item_counts[item] += 1
    core_ok = all(n >= 5 for n in user_counts.values()) and all(n >= 5 for n in item_counts.values())
    return user_counts, item_counts, core_ok

def _reference_filter_kcore(user_items):
    # Menyalin urutan pruning filter_Kcore pada data_process.py rujukan.
    user_items = deepcopy(user_items)
    user_counts, item_counts, core_ok = _reference_core_counts(user_items)
    while not core_ok:
        for user, _ in user_counts.items():
            if user_counts[user] < 5:
                user_items.pop(user)
            else:
                for item in user_items[user]:
                    if item_counts[item] < 5:
                        user_items[user].remove(item)
        user_counts, item_counts, core_ok = _reference_core_counts(user_items)
    return user_items

ref_user_items = _reference_filter_kcore(ref_user_items)
ref_user2id, ref_artist2id, ref_sequences = {}, {}, OrderedDict()
for user, artists in ref_user_items.items():
    if user not in ref_user2id:
        ref_user2id[user] = len(ref_user2id) + 1
    uid = ref_user2id[user]
    ids = []
    for artist in artists:
        if artist not in ref_artist2id:
            ref_artist2id[artist] = len(ref_artist2id) + 1
        ids.append(ref_artist2id[artist])
    ref_sequences[uid] = ids

# Compare mappings and complete ordered sequences with the notebook's pipeline.
current_user2id = {str(int(k)): int(v) for k, v in user2id.items()}
current_artist2id = {str(int(k)): int(v) for k, v in artist2id.items()}
assert current_user2id == ref_user2id, 'User-to-ID mapping differs from reference.'
assert current_artist2id == ref_artist2id, 'Artist-to-ID mapping differs from reference.'
assert list(item_sequences.items()) == list(ref_sequences.items()), 'At least one ordered item sequence differs from reference.'

# Verify core, ID range, unique interactions, time ordering, gap arithmetic, and serialized files.
assert len(item_sequences) == n_users == 1090
assert len(artist2id) == n_artists == 3646
assert sum(map(len, item_sequences.values())) == n_interactions == 52551
assert list(item_sequences) == list(range(1, n_users + 1))
assert sorted(artist2id.values()) == list(range(1, n_artists + 1))
item_frequency = Counter(item for items in item_sequences.values() for item in items)
assert min(map(len, item_sequences.values())) >= 5
assert min(item_frequency.values()) >= 5
for uid, items in item_sequences.items():
    times = time_sequences[uid]
    gaps = gap_sequences[uid]
    assert len(items) == len(times) == len(gaps)
    assert len(items) == len(set(items)), f'Duplicate artist for user {uid}'
    assert all(times[i] <= times[i + 1] for i in range(len(times) - 1)), f'Timestamps out of order for user {uid}'
    assert gaps[0] == 0
    assert gaps[1:] == [times[i] - times[i - 1] for i in range(1, len(times))]

def _serialized_lines(mapping):
    return [f'{uid} ' + ' '.join(map(str, values)) for uid, values in mapping.items()]
assert ITEM_FILE.read_text(encoding='utf-8').splitlines() == _serialized_lines(item_sequences)
assert TIME_FILE.read_text(encoding='utf-8').splitlines() == _serialized_lines(time_sequences)
assert GAP_FILE.read_text(encoding='utf-8').splitlines() == _serialized_lines(gap_sequences)

# Persist raw-ID maps so later versions can prove which item/user each integer ID means.
USER_MAP_FILE = DATA_OUT / 'LastFM_user2id.json'
ARTIST_MAP_FILE = DATA_OUT / 'LastFM_artist2id.json'
USER_MAP_FILE.write_text(json.dumps(ref_user2id, ensure_ascii=False, indent=2, sort_keys=True), encoding='utf-8')
ARTIST_MAP_FILE.write_text(json.dumps(ref_artist2id, ensure_ascii=False, indent=2, sort_keys=True), encoding='utf-8')

def _sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def _git_value(args):
    result = subprocess.run(['git', '-C', str(REPO), *args], capture_output=True, text=True)
    return result.stdout.strip() if result.returncode == 0 else 'unavailable'

package_versions = {}
for package in ('numpy', 'pandas', 'scipy', 'torch', 'tqdm'):
    try:
        package_versions[package] = version(package)
    except PackageNotFoundError:
        package_versions[package] = 'not-installed'

manifest = {
    'audit_status': 'PASS',
    'audit_scope': 'per-user ordered sequence, user/item ID mappings, iterative 5-core, timestamp/gap alignment, serialized files',
    'reference': 'RUCAIBox/CIKM2020-S3Rec data/data_process.py LastFM preprocessing',
    'dataset': 'GroupLens HetRec 2011 Last.fm 2K; timestamps are tag-event timestamps, not listening/scrobble timestamps',
    'counts': {'users': n_users, 'artists': n_artists, 'interactions': n_interactions},
    'id_policy': '1-based IDs assigned by reference traversal order after 5-core filtering',
    'code': {'repository': 'https://github.com/yehjin-shin/BSARec', 'git_commit_at_audit': _git_value(['rev-parse', 'HEAD']), 'git_dirty': _git_value(['status', '--porcelain']) != ''},
    'runtime': {'python': platform.python_version(), 'packages': package_versions, 'cuda_available': bool(torch.cuda.is_available()), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None},
    'sha256': {
        'dataset_zip': _sha256(ZIP_PATH), 'raw_timestamp_file': _sha256(RAW_PATH),
        'LastFM.txt': _sha256(ITEM_FILE), 'LastFM_time.txt': _sha256(TIME_FILE), 'LastFM_gap.txt': _sha256(GAP_FILE),
        'LastFM_user2id.json': _sha256(USER_MAP_FILE), 'LastFM_artist2id.json': _sha256(ARTIST_MAP_FILE),
    },
    'note': 'The commit recorded here proves the earlier training run only if this cell ran in that same repository/runtime; the original saved outputs did not record its commit.'
}
MANIFEST_FILE = DATA_OUT / 'LastFM_preprocessing_manifest.json'
MANIFEST_FILE.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('Exact reference mapping and ordered sequence comparison: PASS (1,090/1,090 users)')
print('5-core, unique artists/user, monotonic times, exact gaps, and serialized row checks: PASS')
print('Saved:', USER_MAP_FILE, ARTIST_MAP_FILE, MANIFEST_FILE)
print('BSARec commit at audit time:', manifest['code']['git_commit_at_audit'])


Exact reference mapping and ordered sequence comparison: PASS (1,090/1,090 users)
5-core, unique artists/user, monotonic times, exact gaps, and serialized row checks: PASS
Saved: /content/BSARec/src/data/LastFM_user2id.json /content/BSARec/src/data/LastFM_artist2id.json /content/BSARec/src/data/LastFM_preprocessing_manifest.json
BSARec commit at audit time: c80bdc0f1a889b78fe9a27e2058389cc84caea84


## E. Jalankan BSARec dengan konfigurasi LastFM paper

Tiap seed memiliki nama run/checkpoint berbeda. Training mengikuti early stopping repo resmi. Jalankan ketiga sel ini satu per satu dan pastikan setiap proses selesai sampai `Test Score`.

In [6]:
!python main.py --data_name LastFM --lr 0.001 --alpha 0.9 --c 3 --num_attention_heads 1 --seed 42 --train_name BSARec_LastFM_final_s42

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026-10-29 15:22:11,370 - Namespace(data_dir='./data/', output_dir='output/', data_name='LastFM', do_eval=False, load_model=None, train_name='BSARec_LastFM_final_s42', num_items=10, num_users=1091, lr=0.001, batch_size=256, epochs=200, no_cuda=False, log_freq=1, patience=10, num_workers=4, seed=42, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, gpu_id='0', variance=5, model_type='BSARec', max_seq_length=50, hidden_size=64, num_hidden_layers=2, hidden_act='gelu', num_attention_heads=1, attention_probs_dropout_pr

In [7]:
!python main.py --data_name LastFM --lr 0.001 --alpha 0.9 --c 3 --num_attention_heads 1 --seed 43 --train_name BSARec_LastFM_final_s43

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026-10-29 15:26:17,597 - Namespace(data_dir='./data/', output_dir='output/', data_name='LastFM', do_eval=False, load_model=None, train_name='BSARec_LastFM_final_s43', num_items=10, num_users=1091, lr=0.001, batch_size=256, epochs=200, no_cuda=False, log_freq=1, patience=10, num_workers=4, seed=43, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, gpu_id='0', variance=5, model_type='BSARec', max_seq_length=50, hidden_size=64, num_hidden_layers=2, hidden_act='gelu', num_attention_heads=1, attention_probs_dropout_pr

In [8]:
!python main.py --data_name LastFM --lr 0.001 --alpha 0.9 --c 3 --num_attention_heads 1 --seed 44 --train_name BSARec_LastFM_final_s44

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026-10-29 15:28:35,238 - Namespace(data_dir='./data/', output_dir='output/', data_name='LastFM', do_eval=False, load_model=None, train_name='BSARec_LastFM_final_s44', num_items=10, num_users=1091, lr=0.001, batch_size=256, epochs=200, no_cuda=False, log_freq=1, patience=10, num_workers=4, seed=44, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, gpu_id='0', variance=5, model_type='BSARec', max_seq_length=50, hidden_size=64, num_hidden_layers=2, hidden_act='gelu', num_attention_heads=1, attention_probs_dropout_pr

## F. Jalankan baseline SASRec

SASRec memakai data, split, dan evaluasi full-item yang sama. Parameter `alpha` dan `c` hanya milik BSARec.

In [9]:
!python main.py --model_type SASRec --data_name LastFM --lr 0.001 --num_attention_heads 1 --seed 42 --train_name SASRec_LastFM_final_s42

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026-10-29 15:31:13,146 - Namespace(data_dir='./data/', output_dir='output/', data_name='LastFM', do_eval=False, load_model=None, train_name='SASRec_LastFM_final_s42', num_items=10, num_users=1091, lr=0.001, batch_size=256, epochs=200, no_cuda=False, log_freq=1, patience=10, num_workers=4, seed=42, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, gpu_id='0', variance=5, model_type='SASRec', max_seq_length=50, hidden_size=64, num_hidden_layers=2, hidden_act='gelu', num_attention_heads=1, attention_probs_dropout_pr

In [10]:
!python main.py --model_type SASRec --data_name LastFM --lr 0.001 --num_attention_heads 1 --seed 43 --train_name SASRec_LastFM_final_s43

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026-10-29 15:32:41,054 - Namespace(data_dir='./data/', output_dir='output/', data_name='LastFM', do_eval=False, load_model=None, train_name='SASRec_LastFM_final_s43', num_items=10, num_users=1091, lr=0.001, batch_size=256, epochs=200, no_cuda=False, log_freq=1, patience=10, num_workers=4, seed=43, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, gpu_id='0', variance=5, model_type='SASRec', max_seq_length=50, hidden_size=64, num_hidden_layers=2, hidden_act='gelu', num_attention_heads=1, attention_probs_dropout_pr

In [11]:
!python main.py --model_type SASRec --data_name LastFM --lr 0.001 --num_attention_heads 1 --seed 44 --train_name SASRec_LastFM_final_s44

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
2026-10-29 15:34:31,650 - Namespace(data_dir='./data/', output_dir='output/', data_name='LastFM', do_eval=False, load_model=None, train_name='SASRec_LastFM_final_s44', num_items=10, num_users=1091, lr=0.001, batch_size=256, epochs=200, no_cuda=False, log_freq=1, patience=10, num_workers=4, seed=44, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, gpu_id='0', variance=5, model_type='SASRec', max_seq_length=50, hidden_size=64, num_hidden_layers=2, hidden_act='gelu', num_attention_heads=1, attention_probs_dropout_pr

## G. Ringkasan metrik test otomatis dari log

Sel ini mengambil metrik dari bagian `Test Score` di log—bukan dari metrik validasi tiap epoch. Jika suatu run belum ada atau gagal, namanya akan ditampilkan sebagai belum tersedia.

In [12]:
import re
import pandas as pd

METRICS = ['HR@5', 'NDCG@5', 'HR@10', 'NDCG@10', 'HR@20', 'NDCG@20']
PAPER = {
    'BSARec': [0.0523, 0.0344, 0.0807, 0.0435, 0.1174, 0.0526],
    'SASRec': [0.0413, 0.0284, 0.0633, 0.0355, 0.0927, 0.0429],
}
OUTPUT_DIR = SRC / 'output'

def read_final_test(model, seed):
    log_path = OUTPUT_DIR / f'{model}_LastFM_final_s{seed}.log'
    if not log_path.exists():
        return None
    log_text = log_path.read_text(encoding='utf-8', errors='ignore')
    marker = '---------------Test Score---------------'
    if marker not in log_text:
        return None
    final_section = log_text.rsplit(marker, 1)[-1]
    result = {}
    for metric in METRICS:
        pattern = re.compile(r"['\"]" + re.escape(metric) + r"['\"]\s*:\s*['\"]?([0-9]+\.[0-9]+)")
        found = pattern.findall(final_section)
        if not found:
            return None
        result[metric] = float(found[-1])
    return result

rows = []
missing = []
for model in ('BSARec', 'SASRec'):
    for seed in (42, 43, 44):
        result = read_final_test(model, seed)
        if result is None:
            missing.append(f'{model} seed {seed}')
            continue
        rows.append({'model': model, 'seed': seed, **result})

if missing:
    print('Run belum memiliki log Test Score:', ', '.join(missing))

if rows:
    detail = pd.DataFrame(rows)
    display(detail.sort_values(['model', 'seed']).round(4))
    summaries = []
    for model in ('BSARec', 'SASRec'):
        model_rows = detail[detail['model'] == model]
        if model_rows.empty:
            continue
        means = model_rows[METRICS].mean()
        stds = model_rows[METRICS].std(ddof=1)
        summary = pd.DataFrame({'Mean': means, 'Std': stds, 'Paper': PAPER[model]})
        summary['Gap vs paper (%)'] = (summary['Mean'] / summary['Paper'] - 1) * 100
        print(f'\n=== {model}: {len(model_rows)} completed seeds ===')
        display(summary.round(4))
        summary.insert(0, 'model', model)
        summary.insert(1, 'metric', summary.index)
        summaries.append(summary.reset_index(drop=True))
    detail.to_csv(OUTPUT_DIR / 'LastFM_final_test_by_seed.csv', index=False)
    if summaries:
        pd.concat(summaries, ignore_index=True).to_csv(OUTPUT_DIR / 'LastFM_final_test_summary.csv', index=False)
else:
    print('Belum ada hasil test yang bisa diringkas. Jalankan sel training dan pastikan log selesai.')

,model,seed,HR@5,NDCG@5,HR@10,NDCG@10,HR@20,NDCG@20
0,BSARec,42,0.0495,0.0333,0.0706,0.0402,0.1101,0.0502
1,BSARec,43,0.0477,0.0353,0.0771,0.0449,0.1147,0.0545
2,BSARec,44,0.0541,0.0356,0.0725,0.0414,0.1083,0.0503
3,SASRec,42,0.0450,0.0337,0.0642,0.0398,0.0881,0.0458
4,SASRec,43,0.0477,0.0334,0.0624,0.0381,0.0982,0.0472
5,SASRec,44,0.0541,0.0381,0.0697,0.0433,0.0945,0.0495



=== BSARec: 3 completed seeds ===


,Mean,Std,Paper,Gap vs paper (%)
HR@5,0.0504,0.0033,0.0523,-3.5692
NDCG@5,0.0347,0.0013,0.0344,0.9690
HR@10,0.0734,0.0033,0.0807,-9.0458
NDCG@10,0.0422,0.0024,0.0435,-3.0651
HR@20,0.1110,0.0033,0.1174,-5.4231
NDCG@20,0.0517,0.0025,0.0526,-1.7744



=== SASRec: 3 completed seeds ===


,Mean,Std,Paper,Gap vs paper (%)
HR@5,0.0489,0.0047,0.0413,18.4826
NDCG@5,0.0351,0.0026,0.0284,23.4742
HR@10,0.0654,0.0038,0.0633,3.3702
NDCG@10,0.0404,0.0027,0.0355,13.8028
HR@20,0.0936,0.0051,0.0927,0.9709
NDCG@20,0.0475,0.0019,0.0429,10.7226


## G.1 Perbandingan berpasangan antar-seed

Sel ini membandingkan BSARec dan SASRec pada seed yang sama. Delta positif berarti BSARec lebih tinggi. Dengan tiga seed, hasil ini deskriptif dan belum cukup untuk klaim signifikansi statistik atau generalisasi.


In [13]:
if 'detail' not in globals() or detail.empty:
    print('Tidak ada tabel metrik lengkap di memori. Jalankan sel ringkasan setelah enam log tersedia.')
else:
    expected_models = {'BSARec', 'SASRec'}
    assert set(detail['model']) == expected_models
    assert sorted(detail['seed'].unique().tolist()) == [42, 43, 44]
    assert len(detail) == 6 and not detail.duplicated(['model', 'seed']).any()
    paired_rows = []
    for metric in METRICS:
        deltas = []
        for seed in (42, 43, 44):
            b = float(detail.loc[(detail['model'] == 'BSARec') & (detail['seed'] == seed), metric].iloc[0])
            s = float(detail.loc[(detail['model'] == 'SASRec') & (detail['seed'] == seed), metric].iloc[0])
            deltas.append(b - s)
        paired_rows.append({
            'metric': metric, 'mean_delta_BSARec_minus_SASRec': sum(deltas) / len(deltas),
            'BSARec_wins': sum(x > 0 for x in deltas), 'ties': sum(x == 0 for x in deltas),
            'SASRec_wins': sum(x < 0 for x in deltas),
        })
    paired = pd.DataFrame(paired_rows)
    display(paired.round(5))
    paired.to_csv(OUTPUT_DIR / 'LastFM_paired_seed_deltas.csv', index=False)


,metric,mean_delta_BSARec_minus_SASRec,BSARec_wins,ties,SASRec_wins
0,HR@5,0.00150,1,2,0
1,NDCG@5,-0.00033,1,0,2
2,HR@10,0.00797,3,0,0
3,NDCG@10,0.00177,2,0,1
4,HR@20,0.01743,3,0,0
5,NDCG@20,0.00417,3,0,0


## 1. Arsip artefak ke Google Drive

**Status saat salinan ini dibuat: BELUM DIEKSEKUSI.** Jalankan di runtime Colab yang masih memiliki keenam log/checkpoint dan file CSV/JSON. Sel membuat satu folder Drive, menyalin hanya artefak yang diminta (tanpa data mentah atau `LastFM*.txt`), lalu menulis SHA-256 semua file yang disalin ke `ARTIFACTS_SHA256.txt`. File checksum tidak memasukkan dirinya sendiri.


In [ ]:
from google.colab import drive
from datetime import datetime
import hashlib, shutil
from pathlib import Path
drive.mount('/content/drive')

RUN_PREFIXES = [f'BSARec_LastFM_final_s{s}' for s in (42, 43, 44)] + [f'SASRec_LastFM_final_s{s}' for s in (42, 43, 44)]
repo_files = [p for p in REPO.rglob('*') if p.is_file()]
archive_sources = []
for _prefix in RUN_PREFIXES:
    _logs = sorted(p for p in repo_files if p.name.startswith(_prefix) and p.suffix.lower() == '.log')
    _checkpoints = sorted(p for p in repo_files if p.name.startswith(_prefix) and p.suffix.lower() == '.pt')
    assert _logs, f'Log .log tidak ditemukan untuk {_prefix}'
    assert _checkpoints, f'Checkpoint .pt tidak ditemukan untuk {_prefix}'
    archive_sources.extend(_logs)
    archive_sources.extend(_checkpoints)

for _name in (
    'LastFM_final_test_by_seed.csv',
    'LastFM_final_test_summary.csv',
    'LastFM_paired_seed_deltas.csv',
):
    _path = OUTPUT_DIR / _name
    assert _path.is_file(), f'CSV wajib tidak ditemukan: {_path}'
    archive_sources.append(_path)
for _name in (
    'LastFM_preprocessing_manifest.json',
    'LastFM_user2id.json',
    'LastFM_artist2id.json',
):
    _path = DATA_OUT / _name
    assert _path.is_file(), f'JSON wajib tidak ditemukan: {_path}'
    archive_sources.append(_path)

# Hindari duplikasi nama/sumber dan jangan pernah menyalin file sequence atau raw data.
archive_sources = sorted({p.resolve(): p for p in archive_sources}.values(), key=lambda p: p.name)
assert not any(p.name.startswith('LastFM') and p.suffix.lower() == '.txt' for p in archive_sources), 'File LastFM*.txt tidak boleh masuk arsip.'
DRIVE_ARCHIVE_PATH = Path('/content/drive/MyDrive') / ('BSARec_LastFM_Artifacts_' + datetime.now().strftime('%Y%m%d_%H%M%S'))
DRIVE_ARCHIVE_PATH.mkdir(parents=True, exist_ok=False)
copied_files = []
for _src in archive_sources:
    _target = DRIVE_ARCHIVE_PATH / _src.name
    if _target.exists():
        _target = DRIVE_ARCHIVE_PATH / f'{_src.parent.name}_{_src.name}'
    shutil.copy2(_src, _target)
    copied_files.append(_target)

def _sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

_hash_lines = [f'{_sha256(p)}  {p.name}' for p in copied_files]
CHECKSUM_FILE = DRIVE_ARCHIVE_PATH / 'ARTIFACTS_SHA256.txt'
CHECKSUM_FILE.write_text('\n'.join(_hash_lines) + '\n', encoding='utf-8')
print('Folder Drive:', DRIVE_ARCHIVE_PATH)
print('Jumlah file yang disalin dan di-hash:', len(archive_sources))
print('Daftar checksum:', CHECKSUM_FILE)


## 2. Runtime info

**Status saat salinan ini dibuat: BELUM DIEKSEKUSI.** Sel ini harus dijalankan setelah training dalam runtime/repository yang sama untuk mengaitkan metadata ke run lama. Pada runtime baru, hasil hanya mendeskripsikan lingkungan saat audit dijalankan.


In [ ]:
import json, platform, subprocess, sys
from importlib.metadata import version, PackageNotFoundError
import torch
EXPECTED_AUDIT_COMMIT = 'c80bdc0f1a889b78fe9a27e2058389cc84caea84'

def _git(args):
    r = subprocess.run(['git', '-C', str(REPO), *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'git {args} gagal: {r.stderr}')
    return r.stdout.strip()

_HEAD = _git(['rev-parse', 'HEAD'])
_STATUS = _git(['status', '--porcelain'])
_versions = {}
for _pkg in ('torch', 'numpy', 'pandas', 'scipy'):
    try:
        _versions[_pkg] = version(_pkg)
    except PackageNotFoundError:
        _versions[_pkg] = 'not-installed'
_HEAD_COMPARISON = 'SAMA' if _HEAD == EXPECTED_AUDIT_COMMIT else 'BERBEDA'
runtime_info = {
    'git_head': _HEAD,
    'expected_audit_commit': EXPECTED_AUDIT_COMMIT,
    'head_comparison': _HEAD_COMPARISON,
    'git_dirty': bool(_STATUS),
    'git_status_porcelain': _STATUS,
    'python_version': platform.python_version(),
    'package_versions': _versions,
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_INFO_FILE = OUTPUT_DIR / 'LastFM_runtime_info.json'
RUNTIME_INFO_FILE.write_text(json.dumps(runtime_info, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(runtime_info, ensure_ascii=False, indent=2))
print('Disimpan ke:', RUNTIME_INFO_FILE)
print('HEAD vs commit audit:', _HEAD_COMPARISON)


## 3. Statistik timestamp

**Status saat salinan ini dibuat: BELUM DIEKSEKUSI.** Sel berikut menghitung gap bertetangga, gap nonpositif, timestamp pasangan bertetangga yang sama, serta jumlah nilai unik dan lima residu terbanyak dari timestamp modulo satu hari dalam milidetik.


In [ ]:
from collections import Counter
DAY_MS = 86_400_000
_time_rows = [np.asarray(ts, dtype=np.int64) for ts in time_sequences.values()]
_timestamp_values_ms = np.concatenate(_time_rows)
_gap_values_ms = np.concatenate([np.diff(ts) for ts in _time_rows if len(ts) > 1])
_timestamp_residues = np.remainder(_timestamp_values_ms, DAY_MS)
_residue_counts = Counter(map(int, _timestamp_residues))

timestamp_stats = {
    'total_gap': int(_gap_values_ms.size),
    'gap_le_zero_count': int(np.count_nonzero(_gap_values_ms <= 0)),
    'gap_le_zero_pct': float(np.mean(_gap_values_ms <= 0) * 100),
    'equal_adjacent_count': int(np.count_nonzero(_gap_values_ms == 0)),
    'equal_adjacent_pct': float(np.mean(_gap_values_ms == 0) * 100),
    'unique_day_residues': len(_residue_counts),
    'top5_day_residues': _residue_counts.most_common(5),
}
print(f"Total gap bertetangga: {timestamp_stats['total_gap']:,}")
print(f"Gap <= 0: {timestamp_stats['gap_le_zero_count']:,} ({timestamp_stats['gap_le_zero_pct']:.4f}%)")
print(f"Pasangan bertetangga dengan timestamp sama: {timestamp_stats['equal_adjacent_count']:,} ({timestamp_stats['equal_adjacent_pct']:.4f}%)")
print(f"Jumlah nilai unik timestamp modulo satu hari: {timestamp_stats['unique_day_residues']:,}")
print('Lima residu terbanyak (residu_ms, frekuensi):', timestamp_stats['top5_day_residues'])


Jika output sel 3 menunjukkan mayoritas gap bernilai nol, pasangan item dengan timestamp sama tidak memiliki urutan waktu yang dapat dibedakan: stable sort mempertahankan urutan baris file sumber. Ini mengikuti prosedur S3Rec. Karena timestamp file ini adalah waktu pemberian tag artis, bukan waktu scrobble/mendengarkan, LastFM kurang cocok sebagai dataset utama untuk eksperimen time-aware; MovieLens-1M lebih sesuai bila timestamp rating yang dibutuhkan. Kesimpulan ini adalah interpretasi semantik data, bukan hasil perbandingan model notebook.


## 4. Verifikasi loss pada commit audit

**Status saat salinan ini dibuat: BELUM DIEKSEKUSI.** Sel memakai `git show` pada commit audit yang tercatat, lalu mencetak hanya fungsi `calculate_loss` dari kedua file model.

Interpretasi setelah output terverifikasi: BSARec menggunakan cross-entropy multi-kelas pada skor seluruh item; SASRec menggunakan binary cross-entropy pada item positif dan negatif sampel. Jadi selisih hasil adalah perbandingan implementasi repo dengan objective training yang berbeda, bukan efek arsitektur murni.


In [ ]:
import ast, subprocess
AUDIT_COMMIT = 'c80bdc0f1a889b78fe9a27e2058389cc84caea84'
LOSS_FILES = [
    ('BSARec', 'src/model/bsarec.py', 'BSARecModel'),
    ('SASRec', 'src/model/sasrec.py', 'SASRecModel'),
]
loss_audit = {}
for _label, _relative_path, _class_name in LOSS_FILES:
    _show = subprocess.run(
        ['git', '-C', str(REPO), 'show', f'{AUDIT_COMMIT}:{_relative_path}'],
        capture_output=True, text=True
    )
    if _show.returncode != 0:
        raise RuntimeError(f'git show gagal untuk {_relative_path}: {_show.stderr}')
    _source = _show.stdout
    _tree = ast.parse(_source)
    _cls = next((n for n in _tree.body if isinstance(n, ast.ClassDef) and n.name == _class_name), None)
    assert _cls is not None, f'Class {_class_name} tidak ditemukan di {_relative_path}@{AUDIT_COMMIT}'
    _fn = next((n for n in _cls.body if isinstance(n, ast.FunctionDef) and n.name == 'calculate_loss'), None)
    assert _fn is not None, f'calculate_loss tidak ditemukan di {_relative_path}@{AUDIT_COMMIT}'
    _snippet = ast.get_source_segment(_source, _fn)
    assert _snippet
    loss_audit[_label] = _snippet
    print(f'===== {_label}: calculate_loss dari {AUDIT_COMMIT}:{_relative_path} =====')
    print(_snippet)


## 5. Granularitas noise HR

**Status saat salinan ini dibuat: BELUM DIEKSEKUSI.** Satu target test per user membuat resolusi HR sebesar `1/n_test`. Sel ini menghitung selisih mean BSARec−SASRec dalam ekuivalen user, delta per seed, serta sample SD delta antar-seed dalam ekuivalen user. Nilai user-equivalent bersifat perkiraan karena log metrik telah dibulatkan.


In [ ]:
assert 'detail' in globals() and len(detail) == 6, 'Jalankan sel G agar enam hasil log tersedia.'
assert set(detail['model']) == {'BSARec', 'SASRec'}
assert sorted(detail['seed'].unique().tolist()) == [42, 43, 44]
assert not detail.duplicated(['model', 'seed']).any()
n_test = sum(1 for items in item_sequences.values() if len(items) >= 3)
assert n_test > 0
print('n_test (jumlah user/target test):', n_test)
print(f'Resolusi HR terkecil 1/n_test: {1/n_test:.8f} ({100/n_test:.6f} percentage point)')
hr_noise_rows = []
for _metric in ('HR@5', 'HR@10', 'HR@20'):
    _b = detail.loc[detail['model']=='BSARec'].set_index('seed')[_metric].sort_index()
    _s = detail.loc[detail['model']=='SASRec'].set_index('seed')[_metric].sort_index()
    assert list(_b.index) == list(_s.index) == [42, 43, 44]
    _delta = (_b - _s) * n_test
    hr_noise_rows.append({
        'metric': _metric,
        'mean_delta_user_equivalent': float(_delta.mean()),
        'delta_seed42_users': float(_delta.loc[42]),
        'delta_seed43_users': float(_delta.loc[43]),
        'delta_seed44_users': float(_delta.loc[44]),
        'sample_SD_delta_across_seeds_users': float(_delta.std(ddof=1)),
    })
hr_noise = pd.DataFrame(hr_noise_rows)
display(hr_noise.round(4))


## 6. Diagnostik epoch dan early stopping

**Status saat salinan ini dibuat: BELUM DIEKSEKUSI.** Parser membatasi diri pada teks sebelum `---------------Test Score---------------`; baris `Epoch 0` test tidak ikut masuk. Ia melaporkan epoch validasi terakhir, semua epoch yang tie pada maksimum NDCG@20 setelah pembulatan log, skor tersebut, rec_loss terakhir, dan epoch penyimpanan checkpoint dari pesan early stopping.


In [ ]:
import ast, re
DIAGNOSTIC_RUNS = [(model, seed) for model in ('BSARec', 'SASRec') for seed in (42, 43, 44)]
TEST_MARKER = '---------------Test Score---------------'

def _parse_training_log(log_path):
    assert log_path.is_file(), f'Log tidak ditemukan: {log_path}'
    text = log_path.read_text(encoding='utf-8', errors='replace')
    assert TEST_MARKER in text, f'Penanda Test Score tidak ditemukan: {log_path}'
    before_test = text.split(TEST_MARKER, 1)[0]
    validation, losses, saved_epochs, counters = [], [], [], []
    current_validation_epoch = None
    for line in before_test.splitlines():
        match = re.search(r'(\{[^{}]*\})', line)
        if match:
            try:
                record = ast.literal_eval(match.group(1))
            except (ValueError, SyntaxError):
                record = None
            if isinstance(record, dict) and 'Epoch' in record and 'NDCG@20' in record:
                current_validation_epoch = int(record['Epoch'])
                validation.append(record)
            elif isinstance(record, dict) and 'epoch' in record and 'rec_loss' in record:
                losses.append(record)
        if 'Validation score increased' in line and current_validation_epoch is not None:
            saved_epochs.append(current_validation_epoch)
        counter_match = re.search(r'EarlyStopping counter: (\d+) out of (\d+)', line)
        if counter_match and current_validation_epoch is not None:
            counters.append((current_validation_epoch, int(counter_match.group(1)), int(counter_match.group(2))))
    assert validation, f'Tidak ada baris validasi pre-test di {log_path}'
    assert losses, f'Tidak ada rec_loss pre-test di {log_path}'
    best_printed = max(float(r['NDCG@20']) for r in validation)
    best_epochs_printed = [int(r['Epoch']) for r in validation if float(r['NDCG@20']) == best_printed]
    return {
        'last_epoch': int(validation[-1]['Epoch']),
        'best_rounded_epoch_first': best_epochs_printed[0],
        'best_rounded_epochs_tied': best_epochs_printed,
        'best_ndcg20_logged': best_printed,
        'last_rec_loss_epoch': int(losses[-1]['epoch']),
        'last_rec_loss': losses[-1]['rec_loss'],
        'last_checkpoint_saved_epoch': saved_epochs[-1] if saved_epochs else None,
        'last_earlystop_counter': counters[-1] if counters else None,
        'saved_epochs': saved_epochs,
        'counter_events': counters,
    }

diagnostic_rows = []
for _model, _seed in DIAGNOSTIC_RUNS:
    _prefix = f'{_model}_LastFM_final_s{_seed}'
    _result = _parse_training_log(OUTPUT_DIR / f'{_prefix}.log')
    diagnostic_rows.append({'model': _model, 'seed': _seed, **_result})
    print(f"{_prefix}: last_epoch={_result['last_epoch']}; rounded_best_NDCG@20_epoch(s)={_result['best_rounded_epochs_tied']}; score={_result['best_ndcg20_logged']}; last_rec_loss(epoch={_result['last_rec_loss_epoch']})={_result['last_rec_loss']}; checkpoint_saved_epoch={_result['last_checkpoint_saved_epoch']}; last_counter={_result['last_earlystop_counter']}")
diagnostic = pd.DataFrame(diagnostic_rows)

_s42 = next(r for r in diagnostic_rows if r['model']=='BSARec' and r['seed']==42)
if len(_s42['best_rounded_epochs_tied']) > 1 and _s42['last_checkpoint_saved_epoch'] in _s42['best_rounded_epochs_tied']:
    _last_saved = _s42['last_checkpoint_saved_epoch']
    _later_counters = [x for x in _s42['counter_events'] if x[0] > _last_saved]
    print('BSARec seed 42: nilai NDCG@20 pada log dibulatkan sehingga beberapa epoch dapat tampak tie; pesan "Validation score increased" menunjukkan perbandingan internal presisi penuh menyimpan checkpoint pada epoch yang lebih akhir. Counter patience dimulai ulang setelah simpan itu; baris counter setelahnya menjelaskan epoch berhenti.')
    print('Epoch counter setelah checkpoint terakhir:', [(e, c, p) for e, c, p in _later_counters])
else:
    print('BSARec seed 42: pola tie/checkpoint tidak terkonfirmasi oleh log yang tersedia; jangan gunakan penjelasan tie pembulatan.')


## H. Interpretasi dan batas klaim untuk presentasi

- **Data:** GroupLens HetRec 2011 Last.fm 2K; hasil 1.090 user, 3.646 artis, 52.551 interaksi. Sel audit D.1 membandingkan setiap sequence terurut dan mapping ID dengan implementasi preprocessing rujukan, bukan hanya mencocokkan jumlah total. Lokasi manifest dan mapping: folder Google Drive yang dicetak sel 1 setelah arsip berhasil dibuat.
- **Protokol:** validasi menargetkan item kedua terakhir, test menargetkan item terakhir, dan evaluasi full-item memakai mask interaksi sebelumnya sesuai kode resmi BSARec. Hasil enam run memakai seed 42/43/44; jangan campur log run pemetaan lama.
- **Hasil:** laporkan semua seed, mean ± sample SD, dan gap paper sebagai perbandingan deskriptif. Paper bukan distribusi tiga seed, jadi gap bukan uji signifikansi. SASRec di notebook ini adalah baseline yang dijalankan dengan konfigurasi yang dinyatakan, bukan klaim reproduksi semua konfigurasi SASRec di paper.
- **Perbandingan model:** gunakan tabel paired seed di G.1. Hindari klaim umum “BSARec selalu lebih baik”; simpulkan per metrik dan jumlah seed yang menang/seri.
- **Timestamp:** timestamp adalah waktu pemberian tag artis, bukan waktu mendengar musik. Model vanilla tidak mengonsumsi file timestamp/gap, jadi hasil ini bukan eksperimen temporal.
- **Reproduksibilitas kode:** manifest merekam commit/hash saat audit dijalankan. Output training yang tersimpan belum merekam commit repo; bila audit tidak dijalankan pada runtime training yang sama, commit run lama tetap tidak terverifikasi. W&B opsional untuk kurva/log terpusat, bukan syarat agar metrik valid.

- **Perbandingan per metrik:** sel H.1 menghitung jumlah seed menang/seri per metrik berdasarkan G.1, menampilkan selisih HR dalam satuan user beserta variasi antar-seed, lalu menyusun kesimpulan hanya jika didukung output. Jangan menyatakan BSARec selalu lebih baik.
- **Gap paper:** tabel G menunjukkan arah deviasi untuk SASRec dan BSARec. Perlakukan gap sebagai deskriptif; versi library, GPU, dan seed adalah kemungkinan penyebab yang belum diuji.
- **Loss:** pakai caveat objective pada output sel 4; jangan mengatribusikan seluruh selisih ke arsitektur saja.
- **Timestamp:** pakai statistik residu dan gap pada output sel 3. Jika mayoritas gap nol, tie mengikuti urutan baris file karena stable sort sesuai prosedur S3Rec; timestamp tag bukan waktu mendengar sehingga LastFM kurang cocok sebagai dataset utama time-aware.
- **Early stopping:** pakai output sel 6. Bila skor NDCG@20 tercetak tie setelah pembulatan tetapi pesan `Validation score increased` muncul pada epoch berikutnya, penjelasan harus membedakan skor log yang dibulatkan dan skor internal yang dibandingkan early stopping.


In [ ]:
# Narasi H dihitung hanya dari output G, G.1, dan sel analisis 2-6 yang benar-benar dijalankan.
assert 'detail' in globals() and len(detail) == 6, 'Jalankan sel G terlebih dahulu.'
assert 'timestamp_stats' in globals(), 'Jalankan sel 3 terlebih dahulu.'
assert 'loss_audit' in globals(), 'Jalankan sel 4 terlebih dahulu.'
assert 'hr_noise' in globals(), 'Jalankan sel 5 terlebih dahulu.'
assert 'diagnostic' in globals() and len(diagnostic) == 6, 'Jalankan sel 6 terlebih dahulu.'

print('=== Perbandingan berpasangan per metrik ===')
paired_outcomes = {}
for _metric in METRICS:
    _d = []
    for _seed in (42, 43, 44):
        _b = float(detail.loc[(detail['model'] == 'BSARec') & (detail['seed'] == _seed), _metric].iloc[0])
        _s = float(detail.loc[(detail['model'] == 'SASRec') & (detail['seed'] == _seed), _metric].iloc[0])
        _d.append(_b - _s)
    _wins, _ties, _losses = sum(x > 0 for x in _d), sum(x == 0 for x in _d), sum(x < 0 for x in _d)
    paired_outcomes[_metric] = (_wins, _ties, _losses, float(np.mean(_d)))
    print(f"{_metric}: BSARec menang {_wins}, seri {_ties}, SASRec menang {_losses} dari tiga seed; mean delta={np.mean(_d):+.6f}.")
    if _metric.startswith('HR@'):
        _nr = hr_noise.loc[hr_noise['metric'] == _metric].iloc[0]
        print(f"  Mean delta={float(_nr['mean_delta_user_equivalent']):+.3f} user; per seed 42/43/44={float(_nr['delta_seed42_users']):+.3f}/{float(_nr['delta_seed43_users']):+.3f}/{float(_nr['delta_seed44_users']):+.3f} user; SD delta antar-seed={float(_nr['sample_SD_delta_across_seeds_users']):.3f} user.")
if paired_outcomes['HR@20'][0] == 3 and paired_outcomes['NDCG@20'][0] == 3:
    print('BSARec konsisten unggul di HR@20 dan NDCG@20 pada ketiga seed.')
if paired_outcomes['HR@10'][0] == 3:
    print('HR@10 searah: BSARec lebih tinggi pada ketiga seed.')
_hr5 = hr_noise.loc[hr_noise['metric'] == 'HR@5'].iloc[0]
if abs(float(_hr5['mean_delta_user_equivalent'])) <= float(_hr5['sample_SD_delta_across_seeds_users']):
    print('HR@5 setara dalam noise pada tiga seed ini: mean delta user tidak melebihi SD delta antar-seed.')
else:
    print('HR@5: tampilkan mean delta user dan SD antar-seed tanpa menyebut setara dalam noise.')
print('NDCG@5 tetap dilaporkan dengan jumlah seed menang/seri dan mean delta di atas; interpretasinya deskriptif.')

print('=== Perbandingan terhadap angka paper (deskriptif) ===')
paper_gaps = {}
for _model in ('BSARec', 'SASRec'):
    _mean = detail.loc[detail['model'] == _model, METRICS].mean()
    _paper = pd.Series(PAPER[_model], index=METRICS)
    _gap = (_mean / _paper - 1) * 100
    paper_gaps[_model] = _gap
    print(f"{_model} gap per metrik (%): " + ', '.join(f"{m}={_gap[m]:+.2f}%" for m in METRICS))
if all(float(x) > 0 for x in paper_gaps['SASRec']) and sum(float(x) < 0 for x in paper_gaps['BSARec']) >= 5:
    print('SASRec berada di atas angka paper pada semua metrik, sedangkan BSARec sedikit di bawah pada mayoritas metrik.')
print('Kedua perbandingan bersifat deskriptif; penyebab deviasi belum diuji.')

print('=== Caveat objective loss ===')
print('BSARec: CrossEntropy multi-kelas atas skor seluruh item, sesuai calculate_loss yang dicetak pada sel 4.')
print('SASRec: BCE-with-logits pada item positif dan negatif sampel, sesuai calculate_loss yang dicetak pada sel 4.')
print('Objective training berbeda, jadi hasil bukan perbandingan efek arsitektur murni.')

print('=== Caveat timestamp ===')
print(f"Total gap={timestamp_stats['total_gap']:,}; gap <= 0={timestamp_stats['gap_le_zero_count']:,} ({timestamp_stats['gap_le_zero_pct']:.4f}%); pasangan timestamp sama={timestamp_stats['equal_adjacent_count']:,} ({timestamp_stats['equal_adjacent_pct']:.4f}%); residu harian unik={timestamp_stats['unique_day_residues']:,}; lima residu teratas={timestamp_stats['top5_day_residues']}.")
if timestamp_stats['equal_adjacent_pct'] > 50:
    print('Sebagian besar gap bernilai 0; urutan item dengan timestamp sama mengikuti urutan baris file lewat stable sort sesuai prosedur S3Rec. Timestamp tag bukan waktu mendengar, sehingga LastFM kurang cocok untuk eksperimen time-aware.')
else:
    print('Timestamp tag bukan waktu mendengar; gunakan statistik gap/residu di atas untuk membatasi klaim time-aware.')

_s42 = diagnostic.loc[(diagnostic['model'] == 'BSARec') & (diagnostic['seed'] == 42)].iloc[0]
print('=== Early stopping BSARec seed 42 ===')
print('Epoch validasi terakhir=', _s42['last_epoch'], '; epoch dengan NDCG@20 tercetak terbaik=', _s42['best_rounded_epochs_tied'], '; skor tercetak=', _s42['best_ndcg20_logged'], '; epoch checkpoint terakhir disimpan=', _s42['last_checkpoint_saved_epoch'], '; counter terakhir=', _s42['last_earlystop_counter'])
print('Jika skor tercetak tie tetapi pesan "Validation score increased" muncul pada epoch sesudahnya, skor internal berpresisi penuh membaik dan counter di-reset saat checkpoint disimpan. Epoch berhenti dihitung dari checkpoint terakhir, bukan tie pertama pada log yang dibulatkan.')


## Referensi

- Paper BSARec, AAAI 2024: https://ojs.aaai.org/index.php/AAAI/article/view/28747
- Implementasi resmi BSARec: https://github.com/yehjin-shin/BSARec
- Split dan pemuatan data BSARec: https://github.com/yehjin-shin/BSARec/blob/main/src/dataset.py
- Evaluasi full-item BSARec: https://github.com/yehjin-shin/BSARec/blob/main/src/trainers.py
- Dataset HetRec 2011 Last.fm 2K, GroupLens: https://grouplens.org/datasets/hetrec-2011/
- Preprocessing LastFM rujukan (deduplikasi, timestamp order, 5-core, mapping ID): https://github.com/RUCAIBox/CIKM2020-S3Rec/blob/master/data/data_process.py

- Source model pada commit audit: `git show c80bdc0f1a889b78fe9a27e2058389cc84caea84:src/model/bsarec.py` dan `...:src/model/sasrec.py`.
- MovieLens-1M, GroupLens: https://grouplens.org/datasets/movielens/1m/
